# High-scoring sequences and sequence dissimilarity

Compare QA-generated sequences from bVAE and pLM-bVAE (HPO) for MHC-I binding peptides. The oracle score is the mean prediction of the three predictors; the reference threshold is derived from training and validation sequences.

In [1]:
# Count high-scoring QA-generated sequences and measure their nearest-neighbor sequence dissimilarity for MHC-I binding peptides.
# Import libraries for data handling and visualization.
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore")

import matplotlib as mpl
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['axes.unicode_minus'] = False

## Load generated sequences and predictor scores

In [2]:
# Load sequences generated by bVAE and their predicted scores across runs; calculate the mean predictor score.
# Load the data.
plm_res = "...\\pLM-bVAE\\reproduction_paper\\outputs\\"
# bvae
bvae_all_sample1 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa1\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
bvae_all_sample2 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa2\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
bvae_all_sample3 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa3\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
bvae_all_sample4 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa4\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
bvae_all_sample5 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa5\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
bvae_all_sample6 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa6\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
bvae_all_sample7 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa7\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
bvae_all_sample8 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa8\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
bvae_all_sample9 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa9\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
bvae_all_sample10 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa10\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])

# Load the sequences.
# bvae decoded_list
bvae_decoded1 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa1\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
bvae_decoded2 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa2\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
bvae_decoded3 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa3\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
bvae_decoded4 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa4\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
bvae_decoded5 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa5\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
bvae_decoded6 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa6\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
bvae_decoded7 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa7\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
bvae_decoded8 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa8\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
bvae_decoded9 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa9\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
bvae_decoded10 = pd.read_csv(plm_res +'moqa\\peptide\\bvae_moqa10\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])

# Merge decoded sequences with sampled data.
bvae_all_sample1 = pd.concat([bvae_decoded1, bvae_all_sample1], axis=1)
bvae_all_sample2 = pd.concat([bvae_decoded2, bvae_all_sample2], axis=1)
bvae_all_sample3 = pd.concat([bvae_decoded3, bvae_all_sample3], axis=1)
bvae_all_sample4 = pd.concat([bvae_decoded4, bvae_all_sample4], axis=1)
bvae_all_sample5 = pd.concat([bvae_decoded5, bvae_all_sample5], axis=1)
bvae_all_sample6 = pd.concat([bvae_decoded6, bvae_all_sample6], axis=1)
bvae_all_sample7 = pd.concat([bvae_decoded7, bvae_all_sample7], axis=1)
bvae_all_sample8 = pd.concat([bvae_decoded8, bvae_all_sample8], axis=1)
bvae_all_sample9 = pd.concat([bvae_decoded9, bvae_all_sample9], axis=1)
bvae_all_sample10 = pd.concat([bvae_decoded10, bvae_all_sample10], axis=1)

# Combine the ten bVAE runs.
bvae_all_samples = pd.concat([bvae_all_sample1, bvae_all_sample2, bvae_all_sample3, bvae_all_sample4, bvae_all_sample5, bvae_all_sample6, bvae_all_sample7, bvae_all_sample8, bvae_all_sample9, bvae_all_sample10], ignore_index=True)
len(bvae_all_samples)

# Remove whitespace from each amino acid sequence.
bvae_all_samples['seq'] = bvae_all_samples['seq'].str.replace(' ', '')

# Average the three predictor scores in bb_score_avg.
bvae_all_samples['bb_score_avg'] = bvae_all_samples[['bb_score1', 'bb_score2', 'bb_score3']].mean(axis=1)

len(bvae_all_samples)

20000

In [3]:
# Load sequences generated by pLM-bVAE (HPO) and their predicted scores across runs; calculate the mean predictor score.
# esm_650m_opt
esm_650m_opt_all_sample1 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa1\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
esm_650m_opt_all_sample2 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa2\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
esm_650m_opt_all_sample3 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa3\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
esm_650m_opt_all_sample4 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa4\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
esm_650m_opt_all_sample5 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa5\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
esm_650m_opt_all_sample6 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa6\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
esm_650m_opt_all_sample7 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa7\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
esm_650m_opt_all_sample8 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa8\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
esm_650m_opt_all_sample9 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa9\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])
esm_650m_opt_all_sample10 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa10\\model_output\\binary\\all_points_samples.txt', sep=" ", header=None, names=["iter", "bb_score1", "bb_score2", "bb_score3", "moqa_score", "E_norm_reads", "cbf_reads", "E_reads", "fm_score"])

# Load the sequences.
# esm_650m_opt decoded_list
esm_650m_opt_decoded1 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa1\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
esm_650m_opt_decoded2 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa2\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
esm_650m_opt_decoded3 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa3\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
esm_650m_opt_decoded4 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa4\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
esm_650m_opt_decoded5 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa5\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
esm_650m_opt_decoded6 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa6\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
esm_650m_opt_decoded7 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa7\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
esm_650m_opt_decoded8 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa8\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
esm_650m_opt_decoded9 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa9\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])
esm_650m_opt_decoded10 = pd.read_csv(plm_res +'moqa\\peptide\\esm_650m_opt_moqa10\\model_output\\binary\\decoded_list.txt', sep="\t", header=None, names=["seq"])

# Merge decoded sequences with sampled data.
esm_650m_opt_all_sample1 = pd.concat([esm_650m_opt_decoded1, esm_650m_opt_all_sample1], axis=1)
esm_650m_opt_all_sample2 = pd.concat([esm_650m_opt_decoded2, esm_650m_opt_all_sample2], axis=1)
esm_650m_opt_all_sample3 = pd.concat([esm_650m_opt_decoded3, esm_650m_opt_all_sample3], axis=1)
esm_650m_opt_all_sample4 = pd.concat([esm_650m_opt_decoded4, esm_650m_opt_all_sample4], axis=1)
esm_650m_opt_all_sample5 = pd.concat([esm_650m_opt_decoded5, esm_650m_opt_all_sample5], axis=1)
esm_650m_opt_all_sample6 = pd.concat([esm_650m_opt_decoded6, esm_650m_opt_all_sample6], axis=1)
esm_650m_opt_all_sample7 = pd.concat([esm_650m_opt_decoded7, esm_650m_opt_all_sample7], axis=1)
esm_650m_opt_all_sample8 = pd.concat([esm_650m_opt_decoded8, esm_650m_opt_all_sample8], axis=1)
esm_650m_opt_all_sample9 = pd.concat([esm_650m_opt_decoded9, esm_650m_opt_all_sample9], axis=1)
esm_650m_opt_all_sample10 = pd.concat([esm_650m_opt_decoded10, esm_650m_opt_all_sample10], axis=1)

# Combine the ten pLM-bVAE (HPO) runs.
esm_650m_opt_all_samples = pd.concat([esm_650m_opt_all_sample1, esm_650m_opt_all_sample2, esm_650m_opt_all_sample3, esm_650m_opt_all_sample4, esm_650m_opt_all_sample5, esm_650m_opt_all_sample6, esm_650m_opt_all_sample7, esm_650m_opt_all_sample8, esm_650m_opt_all_sample9, esm_650m_opt_all_sample10], ignore_index=True)
len(esm_650m_opt_all_samples)

# Remove whitespace from each amino acid sequence.
esm_650m_opt_all_samples['seq'] = esm_650m_opt_all_samples['seq'].str.replace(' ', '')

# Average the three predictor scores in bb_score_avg.
esm_650m_opt_all_samples['bb_score_avg'] = esm_650m_opt_all_samples[['bb_score1', 'bb_score2', 'bb_score3']].mean(axis=1)

len(esm_650m_opt_all_samples)

19995

## Load training and validation sequences

In [4]:
# Load the training and validation sequences used as the reference set.
# Load the training data.
plm_data = "...\\pLM-bVAE\\reproduction_paper\\data\\"
train_seqs = pd.read_csv(plm_data + "peptide\\bvae\\train.txt", sep="\t", header=None, names=["seq"])
train_seqs["seq"] = train_seqs["seq"].str.replace(" ", "")

# Load the validation data.
dev_seqs = pd.read_csv(plm_data + "peptide\\bvae\\dev.txt", sep="\t", header=None, names=["seq"])
dev_seqs["seq"] = dev_seqs["seq"].str.replace(" ", "")

# Combine the training and validation data.
train_dev_seqs = pd.concat([train_seqs, dev_seqs], ignore_index=True)

## Filter penalized and duplicate sequences

In [5]:
# Identify and exclude sequences assigned the penalty score.
# Count rows with a penalty score of 1e6.
print((bvae_all_samples['bb_score1'] == 1e6).sum())
print((esm_650m_opt_all_samples['bb_score1'] == 1e6).sum())

# Exclude rows with a penalty score of 1e6.
bvae_all_samples_del_penalty = bvae_all_samples[bvae_all_samples['bb_score1'] != 1e6]
esm_650m_opt_all_samples_del_penalty = esm_650m_opt_all_samples[esm_650m_opt_all_samples['bb_score1'] != 1e6]

0
0


In [6]:
# Inspect and remove duplicate generated sequences.
# Count duplicate sequences.
print(bvae_all_samples_del_penalty.duplicated(subset=['seq']).sum())
print(esm_650m_opt_all_samples_del_penalty.duplicated(subset=['seq']).sum())

# Remove duplicate sequences.
bvae_all_samples_del_penalty_unique = bvae_all_samples_del_penalty.drop_duplicates(subset=['seq'])
esm_650m_opt_all_samples_del_penalty_unique = esm_650m_opt_all_samples_del_penalty.drop_duplicates(subset=['seq'])

5166
5258


In [7]:
# Exclude generated sequences present in the training and validation data and remove duplicates.
# Check whether generated bVAE sequences overlap with training and validation sequences.
bvae_in_train_dev = bvae_all_samples_del_penalty['seq'].isin(train_dev_seqs['seq'])
esm_650m_opt_in_train_dev = esm_650m_opt_all_samples_del_penalty_unique['seq'].isin(train_dev_seqs['seq'])
print(f"bVAE generated rows matching training or validation sequences: {bvae_in_train_dev.sum()}")
print(f"Unique pLM-bVAE (HPO) sequences matching training or validation sequences: {esm_650m_opt_in_train_dev.sum()}")

# Remove duplicate sequences.
bvae_all_samples_del_penalty_unique_train_dev = bvae_all_samples_del_penalty_unique[~bvae_in_train_dev]
esm_650m_opt_all_samples_del_penalty_unique_train_dev = esm_650m_opt_all_samples_del_penalty_unique[~esm_650m_opt_in_train_dev]

bVAE generated rows matching training or validation sequences: 1
Unique pLM-bVAE (HPO) sequences matching training or validation sequences: 0


In [8]:
# Inspect the number of retained sequences.
# Check the number of sequences.
print(f"Unique bVAE sequences excluding training and validation sequences: {len(bvae_all_samples_del_penalty_unique_train_dev)}")
print(f"Unique pLM-bVAE (HPO) sequences excluding training and validation sequences: {len(esm_650m_opt_all_samples_del_penalty_unique_train_dev)}")

Unique bVAE sequences excluding training and validation sequences: 14833
Unique pLM-bVAE (HPO) sequences excluding training and validation sequences: 14737


## Calculate the reference oracle score

In [9]:
# Load predictor scores for the training and validation data and calculate their mean.
# Load the training and validation data.
pred_train_val1 = pd.read_csv(plm_res +'fmqa\\pred_parrot_train_val_exp\\pred_parrot_peptide_z1_exp_train_val.txt', sep=" ", header=None, names=["id", "seq", "bb_score"])
pred_train_val2 = pd.read_csv(plm_res +'fmqa\\pred_parrot_train_val_exp\\pred_parrot_peptide_z2_exp_train_val.txt', sep=" ", header=None, names=["id", "seq", "bb_score"])
pred_train_val3 = pd.read_csv(plm_res +'fmqa\\pred_parrot_train_val_exp\\pred_parrot_peptide_z3_exp_train_val.txt', sep=" ", header=None, names=["id", "seq", "bb_score"])
# Calculate the mean predictor score for the training and validation data.
pred_train_val1['bb_score_avg'] = (pred_train_val1['bb_score'] + pred_train_val2['bb_score'] + pred_train_val3['bb_score']) / 3

In [10]:
# Find the reference score corresponding to the tenth-ranked training and validation sequence.
# Find the tenth-ranked training and validation score.
pred_train_val_bb_score_top10 = pred_train_val1.sort_values(by='bb_score_avg', ascending=True).iloc[9]['bb_score_avg']
print(pred_train_val_bb_score_top10)
# Convert the natural-log score back to its original scale.
print(np.exp(pred_train_val_bb_score_top10))

0.274635882
1.3160513896148394


In [11]:
# Count generated sequences meeting the reference score threshold.
# Count generated sequences at or below the reference score.
bvae_top10_count = (bvae_all_samples_del_penalty_unique_train_dev['bb_score_avg'] <= pred_train_val_bb_score_top10).sum()
esm_top10_count = (esm_650m_opt_all_samples_del_penalty_unique_train_dev['bb_score_avg'] <= pred_train_val_bb_score_top10).sum()
print(f"bVAE sequences with oracle scores at or below the reference threshold: {bvae_top10_count}")
print(f"pLM-bVAE (HPO) sequences with oracle scores at or below the reference threshold: {esm_top10_count}")

bVAE sequences with oracle scores at or below the reference threshold: 3976
pLM-bVAE (HPO) sequences with oracle scores at or below the reference threshold: 5635


In [12]:
# Select generated sequences meeting the reference score threshold.
# Select generated sequences at or below the reference score.
bvae_top10 = bvae_all_samples_del_penalty_unique_train_dev[bvae_all_samples_del_penalty_unique_train_dev['bb_score_avg'] <= pred_train_val_bb_score_top10]
esm_top10 = esm_650m_opt_all_samples_del_penalty_unique_train_dev[esm_650m_opt_all_samples_del_penalty_unique_train_dev['bb_score_avg'] <= pred_train_val_bb_score_top10]

## Measure nearest-neighbor sequence dissimilarity

In [13]:
# Calculate the nearest-neighbor Levenshtein distance to the training and validation sequences for each selected sequence.
# Calculate distances from each selected bVAE sequence to the training and validation sequences.
# Store the minimum distance in the bVAE lev_distance column.
import Levenshtein
from tqdm import tqdm
tqdm.pandas()
def calculate_lev_distance(seq):
    distances = pred_train_val1['seq'].apply(lambda x: Levenshtein.distance(seq, x))
    return distances.min()
# Calculate nearest-neighbor distances for selected bVAE sequences with a progress bar.
bvae_top10['lev_distance'] = bvae_top10['seq'].progress_apply(calculate_lev_distance)
# Calculate distances from each selected pLM-bVAE sequence to the training and validation sequences.
# Store the minimum distance in the pLM-bVAE lev_distance column.
esm_top10['lev_distance'] = esm_top10['seq'].progress_apply(calculate_lev_distance)

  0%|          | 0/3976 [00:00<?, ?it/s]

  0%|          | 16/3976 [00:00<00:25, 156.90it/s]

  1%|          | 32/3976 [00:00<00:31, 124.77it/s]

  1%|          | 45/3976 [00:00<00:31, 123.47it/s]

  1%|▏         | 58/3976 [00:00<00:31, 123.15it/s]

...

 99%|█████████▉| 5602/5635 [00:59<00:00, 78.18it/s]

100%|█████████▉| 5611/5635 [00:59<00:00, 79.92it/s]

100%|█████████▉| 5621/5635 [00:59<00:00, 83.99it/s]

100%|█████████▉| 5633/5635 [00:59<00:00, 92.29it/s]

100%|██████████| 5635/5635 [01:00<00:00, 93.89it/s]

In [14]:
# Summarize the nearest-neighbor Levenshtein distances.
print(bvae_top10['lev_distance'].describe())
print(esm_top10['lev_distance'].describe())

count    3976.000000
mean        3.468310
std         0.559876
min         1.000000
25%         3.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: lev_distance, dtype: float64
count    5635.000000
mean        3.583496
std         0.544026
min         1.000000
25%         3.000000
50%         4.000000
75%         4.000000
max         5.000000
Name: lev_distance, dtype: float64
